Preprocess and load old model

In [68]:
from torchvision.io import decode_image
from torchvision import datasets, transforms
from torchvision.models import mnasnet0_5, MNASNet0_5_Weights
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

device = torch.device("cuda")



# Step 1: Initialize model with the best available weights
weights = MNASNet0_5_Weights.DEFAULT
model = mnasnet0_5(weights=weights)

model.classifier = nn.Sequential(
    nn.Conv1d(128,128,8, padding="valid"),
    nn.Dropout(0.25),
    nn.AdaptiveAvgPool1d(20),
    nn.Dropout(0.5),
    nn.Linear(20, 128),
    nn.ReLU(),
    nn.Dropout(0.25),
    nn.Linear(128, 8),  # 8 Classes in DOES
    nn.Flatten(),
    nn.Softmax(dim=1)
)


# Step 2: Initialize the inference transforms
preprocess = transforms.Compose([
    transforms.RandomHorizontalFlip(.5),
    transforms.RandomRotation(degrees=[1,30]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # Some common mean RGB values
                         std=[0.229, 0.224, 0.225])  # Common stddevs for RGB values

])
# Step 3: Apply inference preprocessing transforms
train_dataset = datasets.ImageFolder("DOES", transform=preprocess)
test_dataset = datasets.ImageFolder("TEST", transform=preprocess)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=True,pin_memory=True)






Test Function

In [69]:
def get_accuracy_and_loss(model, loader, criterion):
  model.eval()
  my_loss = 0
  with torch.no_grad():
    correct = 0
    for data, target in loader:
      data, target = data.to(device), target.to(device)
      output = model(data)
      pred = output.argmax(dim=1)
      correct += pred.eq(target).sum().item()
      my_loss += criterion(output, target).item()
  return correct/len(loader.dataset), my_loss/len(loader.dataset)

Run test

In [70]:
import torch.nn.functional as F
import torch.optim as optim
best_val_loss = 1000000
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
model.to(device)

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []
time = 0

for epoch in range(1000):
    model.train()
    train_loss = 0
    correct = 0
    total_count = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)

        train_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total_count += data.size(0)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} done.")
    #Train Accuracy
    train_accuracy = correct / total_count
    train_accuracies.append(train_accuracy)
    print(f"Train accuracy: {train_accuracy}")

    #Train Loss
    train_loss = train_loss / total_count
    train_losses.append(train_loss)
    print(f"Train loss: {train_loss}")

    #Validation Accuracy and Loss
    val_accuracy, val_loss = get_accuracy_and_loss(model, test_loader, criterion)

    #Accuracy
    print(f"Val accuracy: {val_accuracy}")
    val_accuracies.append(val_accuracy)

    #Loss
    print(f"Val loss: {val_loss}")
    val_losses.append(val_loss)

    if val_loss <= best_val_loss:
        best_val_loss = val_loss
        torch.save({'model_state_dict': model.state_dict(),
            "val_loss": val_loss,
            "train_losses":train_losses,
            "val_losses":val_losses},"best_model_extra_conv.pth")
        time = 0
    else:
        if time > 5:
            break
        else:
            time +=1

KeyboardInterrupt: 